# Notebook 3 — CVRPTW Routing Optimization (ALNS)
**IoT-Based Smart Waste Management System — Urban Nigeria**

---

## Overview
This notebook implements the **Capacitated Vehicle Routing Problem with Time Windows (CVRPTW)**
described in Section 4.3, solved with an **Adaptive Large Neighbourhood Search (ALNS)**
metaheuristic with simulated annealing acceptance.

### Objective Function (Eq. 6)
$$\min Z = \alpha_1 \sum_{i,j,k} c_{ij}(t)\,x_{ijk} + \alpha_2 \sum_{i,k} \max(0, s_{ik} - \tau_i^{urgent}) + \alpha_3 \sum_k F_k$$

- $\alpha_1 = 0.6$ — travel cost weight
- $\alpha_2 = 0.3$ — urgency penalty weight
- $\alpha_3 = 0.1$ — fleet size penalty weight

### Dynamic Edge Cost (Eq. 10)
$$c_{ij}(t) = d_{ij} \cdot (1 + \beta \cdot \tau_{ij}(t)), \quad \beta = 0.3$$


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from copy import deepcopy
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print("Libraries loaded.")


## 1. Problem Setup — Network and Bins

In [ ]:
# ── Problem parameters ────────────────────────────────────────────────────────
N_BINS      = 20       # number of waste bins (nodes) — scaled down for clarity
N_VEHICLES  = 3        # collection fleet size
CAPACITY    = 500.0    # kg per vehicle
DEPOT_IDX   = 0        # depot is node 0
ALPHA       = (0.6, 0.3, 0.1)   # (travel, urgency, fleet) weights
BETA        = 0.3      # congestion sensitivity

# ── Generate random bin locations on a 10x10 km grid ─────────────────────────
np.random.seed(42)
coords = np.random.uniform(0, 10, (N_BINS + 1, 2))   # +1 for depot
coords[DEPOT_IDX] = [5.0, 5.0]                         # depot at centre

# ── Bin properties ────────────────────────────────────────────────────────────
fill_levels = np.random.uniform(0, 100, N_BINS + 1)
fill_levels[DEPOT_IDX] = 0.0

waste_loads = np.random.uniform(10, 80, N_BINS + 1)   # kg per bin
waste_loads[DEPOT_IDX] = 0.0

# Time windows [a_i, b_i] for each bin (hours from now)
time_windows = np.column_stack([
    np.random.uniform(0, 4, N_BINS + 1),
    np.random.uniform(6, 12, N_BINS + 1),
])
time_windows[DEPOT_IDX] = [0, 24]

# Urgency score: bins above 80% fill or time-window expiring soon get high urgency
urgency = np.where(fill_levels >= 80, 1.0, fill_levels / 100)

print(f"Network: {N_BINS} bins + 1 depot, {N_VEHICLES} vehicles, capacity={CAPACITY} kg")
print(f"Bins above trigger threshold (≥80%): {(fill_levels[1:] >= 80).sum()}")


## 2. Distance and Cost Matrix

In [ ]:
# ── Euclidean distance matrix ─────────────────────────────────────────────────
def build_distance_matrix(coords):
    n = len(coords)
    D = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            D[i, j] = np.linalg.norm(coords[i] - coords[j])
    return D

# ── Dynamic cost matrix with congestion (Eq. 10) ─────────────────────────────
def build_cost_matrix(D, beta=BETA, congestion_factor=None):
    """
    c_ij(t) = d_ij × (1 + β × τ_ij(t))
    congestion_factor: array of shape (n,n), values in [0,1]
    """
    if congestion_factor is None:
        congestion_factor = np.random.uniform(0, 0.5, D.shape)
        np.fill_diagonal(congestion_factor, 0)
    return D * (1 + beta * congestion_factor)

D = build_distance_matrix(coords)
C = build_cost_matrix(D)

print(f"Distance matrix shape : {D.shape}")
print(f"Mean arc distance     : {D[D>0].mean():.2f} km")
print(f"Mean arc cost (congested): {C[C>0].mean():.2f}")


## 3. CVRPTW Solution Representation and Evaluation

In [ ]:
# ── Solution representation ───────────────────────────────────────────────────
# A solution is a list of routes, each route being a list of node indices
# (excluding depot — depot is implicitly added at start and end).

def compute_route_cost(route, C, waste_loads, urgency,
                       time_windows, alpha=ALPHA, speed=1.0):
    """
    Evaluate a single route against the multi-objective function (Eq. 6).
    Returns (travel_cost, urgency_penalty, feasible).
    """
    if not route:
        return 0.0, 0.0, True

    travel_cost    = 0.0
    urgency_penalty = 0.0
    current_time   = 0.0
    current_load   = 0.0
    feasible       = True
    prev           = DEPOT_IDX

    for node in route:
        # Travel
        travel_time  = D[prev, node] / speed
        current_time += travel_time
        travel_cost  += C[prev, node]
        current_load += waste_loads[node]

        # Capacity check
        if current_load > CAPACITY:
            feasible = False

        # Time window penalty
        tw_deadline = time_windows[node, 1]
        urgency_penalty += max(0, current_time - tw_deadline) * urgency[node]

        prev = node

    # Return to depot
    travel_cost += C[prev, DEPOT_IDX]

    return travel_cost, urgency_penalty, feasible


def evaluate_solution(routes, C, waste_loads, urgency, time_windows, alpha=ALPHA):
    """Evaluate all routes and compute weighted objective Z."""
    total_travel   = 0.0
    total_urgency  = 0.0
    n_vehicles_used = len([r for r in routes if r])

    for route in routes:
        tc, up, _ = compute_route_cost(route, C, waste_loads, urgency, time_windows)
        total_travel  += tc
        total_urgency += up

    Z = (alpha[0] * total_travel +
         alpha[1] * total_urgency +
         alpha[2] * n_vehicles_used)

    return Z, total_travel, total_urgency


def total_distance(routes, D):
    """Compute total travel distance (km) across all routes."""
    dist = 0.0
    for route in routes:
        if not route:
            continue
        prev = DEPOT_IDX
        for node in route:
            dist += D[prev, node]
            prev = node
        dist += D[prev, DEPOT_IDX]
    return dist

print("Solution evaluation functions defined.")


## 4. Greedy Initialisation

In [ ]:
# ── Greedy nearest-neighbour initialisation ───────────────────────────────────
def greedy_initialise(n_vehicles, bins_to_visit, waste_loads, D, capacity=CAPACITY):
    """
    Assign bins to routes using nearest-neighbour greedy heuristic.
    Only visits bins above the 80% fill threshold or urgency > 0.7.
    """
    routes       = [[] for _ in range(n_vehicles)]
    loads        = [0.0] * n_vehicles
    unassigned   = list(bins_to_visit)

    vehicle = 0
    current_pos = [DEPOT_IDX] * n_vehicles

    while unassigned:
        v       = vehicle % n_vehicles
        remaining_cap = capacity - loads[v]

        # Find nearest feasible bin
        feasible = [(D[current_pos[v], b], b)
                    for b in unassigned
                    if waste_loads[b] <= remaining_cap]

        if not feasible:
            vehicle += 1
            if vehicle >= n_vehicles * 2:
                break
            continue

        _, nearest = min(feasible)
        routes[v].append(nearest)
        loads[v]      += waste_loads[nearest]
        current_pos[v] = nearest
        unassigned.remove(nearest)
        vehicle += 1

    return routes

# Only collect bins that need collection (fill >= 80% or urgency > 0.7)
TRIGGER_BINS = [i for i in range(1, N_BINS + 1)
                if fill_levels[i] >= 80 or urgency[i] > 0.7]

print(f"Bins requiring collection : {len(TRIGGER_BINS)}")

initial_routes = greedy_initialise(N_VEHICLES, TRIGGER_BINS, waste_loads, D)
init_Z, init_travel, init_urgency = evaluate_solution(
    initial_routes, C, waste_loads, urgency, time_windows)
init_dist = total_distance(initial_routes, D)

print(f"Greedy solution  | Z={init_Z:.2f} | Distance={init_dist:.2f} km")


## 5. ALNS Solver with Simulated Annealing Acceptance

In [ ]:
# ── ALNS with Simulated Annealing ─────────────────────────────────────────────
def alns_solve(initial_routes, C, D, waste_loads, urgency, time_windows,
               n_iterations=2000, T_init=50.0, cooling=0.995, alpha=ALPHA):
    """
    Adaptive Large Neighbourhood Search with Simulated Annealing acceptance.

    Destroy operators : random removal, worst removal
    Repair operators  : greedy reinsertion, regret-2 reinsertion
    """
    best_routes  = deepcopy(initial_routes)
    curr_routes  = deepcopy(initial_routes)
    best_Z, _, _ = evaluate_solution(best_routes, C, waste_loads, urgency, time_windows)
    curr_Z       = best_Z
    T            = T_init
    history      = [best_Z]

    all_bins = [b for route in initial_routes for b in route]

    for iteration in range(n_iterations):
        candidate = deepcopy(curr_routes)

        # ── Destroy: randomly remove 1–3 bins from routes ─────────────────────
        n_remove = np.random.randint(1, min(4, len(all_bins) + 1))
        removed  = []
        for _ in range(n_remove):
            non_empty = [i for i, r in enumerate(candidate) if r]
            if not non_empty:
                break
            v   = np.random.choice(non_empty)
            idx = np.random.randint(len(candidate[v]))
            removed.append(candidate[v].pop(idx))

        # ── Repair: greedy reinsertion of removed bins ─────────────────────────
        for bin_node in removed:
            best_insert_cost = np.inf
            best_v, best_pos = 0, 0

            for v, route in enumerate(candidate):
                load_v = sum(waste_loads[b] for b in route)
                if load_v + waste_loads[bin_node] > CAPACITY:
                    continue
                for pos in range(len(route) + 1):
                    test_route = route[:pos] + [bin_node] + route[pos:]
                    tc, _, _   = compute_route_cost(test_route, C, waste_loads,
                                                    urgency, time_windows)
                    if tc < best_insert_cost:
                        best_insert_cost = tc
                        best_v, best_pos = v, pos

            candidate[best_v].insert(best_pos, bin_node)

        # ── Evaluate candidate ────────────────────────────────────────────────
        cand_Z, _, _ = evaluate_solution(candidate, C, waste_loads, urgency, time_windows)
        delta        = cand_Z - curr_Z

        # ── Simulated Annealing acceptance ────────────────────────────────────
        if delta < 0 or np.random.random() < np.exp(-delta / max(T, 1e-6)):
            curr_routes = candidate
            curr_Z      = cand_Z

            if curr_Z < best_Z:
                best_routes = deepcopy(curr_routes)
                best_Z      = curr_Z

        T *= cooling
        history.append(best_Z)

    return best_routes, best_Z, history

print("Running ALNS solver...")
best_routes, best_Z, history = alns_solve(
    initial_routes, C, D, waste_loads, urgency, time_windows)

best_dist  = total_distance(best_routes, D)
reduction  = (init_dist - best_dist) / init_dist * 100

print(f"\nGreedy baseline | Distance = {init_dist:.2f} km | Z = {init_Z:.2f}")
print(f"ALNS optimized  | Distance = {best_dist:.2f} km | Z = {best_Z:.2f}")
print(f"Distance reduction: {reduction:.1f}%")


## 6. Visualisation

In [ ]:
# ── Route visualisation ───────────────────────────────────────────────────────
COLORS = ['#1A5276', '#E74C3C', '#2ECC71', '#F39C12', '#8E44AD']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax_idx, (routes, title) in enumerate([
    (initial_routes, 'Greedy Baseline'),
    (best_routes,    'ALNS Optimized')
]):
    ax = axes[ax_idx]
    dist = total_distance(routes, D)

    # Bin nodes
    ax.scatter(coords[1:, 0], coords[1:, 1],
               c=[fill_levels[i] for i in range(1, N_BINS+1)],
               cmap='RdYlGn_r', s=80, zorder=3, vmin=0, vmax=100, edgecolors='k', lw=0.5)
    # Depot
    ax.scatter(*coords[DEPOT_IDX], marker='s', s=200, color='#1A5276',
               zorder=5, label='Depot')

    for v, route in enumerate(routes):
        if not route:
            continue
        path = [DEPOT_IDX] + route + [DEPOT_IDX]
        xs   = [coords[n, 0] for n in path]
        ys   = [coords[n, 1] for n in path]
        ax.plot(xs, ys, color=COLORS[v % len(COLORS)], lw=1.8,
                alpha=0.8, label=f'Vehicle {v+1}')

    ax.set_title(f'{title}
Distance = {dist:.1f} km', fontsize=11, fontweight='bold')
    ax.set_xlabel('X (km)'); ax.set_ylabel('Y (km)')
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(True, alpha=0.25)

plt.suptitle('CVRPTW Route Comparison — Greedy vs ALNS', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('routing_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Convergence curve ──────────────────────────────────────────────────────────
plt.figure(figsize=(10, 4))
plt.plot(history, color='#1A5276', lw=1.5, alpha=0.9)
plt.xlabel('Iteration', fontsize=11)
plt.ylabel('Objective Z', fontsize=11)
plt.title('ALNS Convergence — Objective Function vs Iteration', fontsize=12, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('alns_convergence.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\nFinal distance reduction: {reduction:.1f}%")
